# Section 2.3 — The Hydrogen Atom (Weinberg §2.3)

**What this section claims**
- For V = −e²/r the radial equation with E < 0, ρ = 2κr (κ = √(−2μE)/ℏ), u = ρ^{l+1}e^{−ρ/2}w(ρ) gives a power series whose coefficients obey a_{j+1} = a_j (j+l+1−n)/[(j+1)(j+2l+2)] with n ≡ μe²/(ℏ²κ).
- The series must terminate (otherwise w ~ e^ρ ⇒ u ~ e^{+ρ/2}: non-normalizable), so n = 1, 2, … and E_n = −μe⁴/(2ℏ²n²) = −Ryd/n².
- n = n′ + l + 1 with n′ = 0,…, n−1 and |m| ≤ l gives degeneracy Σ(2l+1) = n².
- R_nl ∝ ρ^l e^{−ρ/2} L_{n−l−1}^{2l+1}(ρ) with ρ = 2r/(na₀) — associated Laguerre polynomials.
- Radiative transitions via the odd-parity vector x: Δl = ±1, Δm = 0, ±1 (parity and tensor-rank selection rules).


## Series solution and termination → E_n = −μe⁴/(2ℏ²n²) (symbolic)

w satisfies ρw″ + (2l+2−ρ)w′ + (n−l−1)w = 0. Verify that the recurrence-generated truncated series is an exact solution at integer n:


In [ ]:
import sympy as sp

rho, n, l = sp.symbols('rho n l', positive=True, integer=True)
a = sp.Function('a')
def series_solution(nv, lv):
    jmax = nv - lv - 1                     # termination index n'
    coeffs = [sp.Integer(1)]
    for j in range(jmax):
        coeffs.append(sp.simplify(coeffs[-1]*(j+lv+1-nv)/((j+1)*(j+2*lv+2))))
    w = sum(c*rho**j for j, c in enumerate(coeffs))
    residual = sp.simplify(rho*sp.diff(w, rho, 2) + (2*lv+2-rho)*sp.diff(w, rho) + (nv-lv-1)*w)
    return w, residual

for nv, lv in [(1,0), (2,0), (3,1), (4,2)]:
    w, res = series_solution(nv, lv)
    print(f"n={nv}, l={lv}: w = {sp.expand(w)}  | residual of w-eq: {res}")

j = sp.symbols('j', positive=True, integer=True)
ratio = sp.simplify((j+l+1-n)/((j+1)*(j+2*l+2)))
print()
print("large-j ratio a_(j+1)/a_j ->", sp.limit(ratio, j, sp.oo), " ~ 1/j  (e^rho tail -> u ~ e^{+rho/2}, not normalizable)")
print("termination forces n = n' + l + 1 = 1,2,... and  kappa = mu e^2/(hbar^2 n)  =>  E_n = -hbar^2 kappa^2/2mu = -mu e^4/(2 hbar^2 n^2)")

## R_nl in associated Laguerre polynomials (symbolic verification)

The polynomial part of w is (up to normalization) the associated Laguerre polynomial L_{n−l−1}^{2l+1}(ρ), which satisfies exactly the w-equation at integer n:


In [ ]:
j = sp.symbols('j', positive=True, integer=True)
rho_ = sp.symbols('rho_', positive=True)
for nv in range(1, 4):
    for lv in range(nv):
        Lag = sp.assoc_laguerre(nv-lv-1, 2*lv+1, rho_)
        res = sp.simplify(rho_*sp.diff(Lag, rho_, 2) + (2*lv+2-rho_)*sp.diff(Lag, rho_) + (nv-lv-1)*Lag)
        print(f"n={nv}, l={lv}: residual of Laguerre eq = {res}")

# normalization:  R_nl = N rho^l e^{-rho/2} L_{n-l-1}^{2l+1}(rho),  N^2 = (2/(n a0))^3 (n-l-1)!/(2n (n+l)!)
print()
print("normalization N^2 = (2/(n a0))^3 (n-l-1)! / [2n (n+l)!]  (verified numerically next)")

## Numerics: normalization ∫r²|R|²dr = 1 and the Bohr spectrum


In [ ]:
import numpy as np, sys
sys.path.insert(0, '..')
from weinberg_constants import e, hbar, me, Ryd, a0
from scipy.integrate import quad
from scipy.special import assoc_laguerre, factorial

def R_nl(n, l, r):
    rho = 2*r/(n*a0)
    N = np.sqrt((2/(n*a0))**3 * factorial(n-l-1)/(2*n*factorial(n+l)))
    return N * rho**l * np.exp(-rho/2) * assoc_laguerre(rho, n-l-1, 2*l+1)

for n, l in [(1,0), (2,0), (2,1), (3,2)]:
    norm, _ = quad(lambda r: r**2*R_nl(n,l,r)**2, 0, 60*a0, epsabs=1e-12)
    print(f"n={n}, l={l}:  int r^2 |R|^2 dr = {norm:.10f}   (expect 1)")

# Bohr spectrum against constants
for n in range(1, 5):
    print(f"E_{n} = {-(Ryd/n**2)/1.602e-12:8.4f} eV")

## Numerics: shooting-method check of E₁ and E₂ (independent of the analytic solution)


In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

def shoot(E_eV, l, rmax=40*a0):
    E = E_eV*1.602e-12
    def rhs(r, y):
        return [y[1], (l*(l+1)/r**2 - 2*me/hbar**2*(E + e**2/r))*y[0]]
    eps = 1e-3*a0
    sol = solve_ivp(rhs, [eps, rmax], [eps**(l+1), (l+1)*eps**l],
                    method='DOP853', rtol=1e-10, atol=1e-14)
    return sol.y[0, -1]

# the tail at rmax flips sign as E crosses a bound-state eigenvalue
targets = {(1,0): -13.6, (2,0): -3.4, (2,1): -3.4}
for (n, l), guess in targets.items():
    E_found = brentq(lambda E: shoot(E, l), guess*1.1, guess*0.9, xtol=1e-6)
    E_exact = -(Ryd/n**2)/1.602e-12
    print(f"n={n}, l={l}: shooting E = {E_found:8.4f} eV   analytic = {E_exact:8.4f} eV   diff = {abs(E_found-E_exact):.1e} eV")

## Numerics: radial probability densities and the n² degeneracy


In [ ]:
import matplotlib.pyplot as plt
r = np.linspace(0, 30*a0, 600)
fig, ax = plt.subplots(figsize=(7, 4))
for n, l, lbl in [(1,0,'1s'), (2,0,'2s'), (2,1,'2p')]:
    ax.plot(r/a0, (r*R_nl(n,l,r))**2 * a0, label=lbl)
ax.set_xlabel('r / a0'); ax.set_ylabel('r^2 |R|^2  (x a0)')
ax.set_title('radial probability: 2s has an inner lobe + node at 2 a0')
ax.legend(); plt.tight_layout(); plt.show()

print("degeneracy of level n:  sum_{l=0}^{n-1} (2l+1) = n^2")
print("n=1: 1 state | n=2: 4 states | n=3: 9 states")

## My summary
*(fill in — own words, ≤ 5 sentences)*

## Gaps
*(questions to raise in review)*
